# Batch 07 · Exp01 · Perbandingan model v3
Jalankan semua sel pada CPU (lokal maupun Colab). Secara lokal notebook otomatis membaca `analysis_sessions/dataset_fusion/output`.

In [ ]:
import sys
from pathlib import Path

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/Penelitian-Adaptive-IDE')
    DATASET_ROOT = PROJECT_ROOT / 'Dataset'
else:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'analysis_sessions/dataset_fusion/output').exists()), cwd)
    DATASET_ROOT = PROJECT_ROOT / 'analysis_sessions/dataset_fusion/output'
OUTPUT_ROOT = PROJECT_ROOT / 'models/batch-07/exp01/outputs'
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATASET_ROOT:', DATASET_ROOT)


## Reader dataset resmi dan fitur sensor

In [ ]:
import hashlib
import json
import platform
import shutil
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import numpy as np
import pandas as pd
import sklearn
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.preprocessing import RobustScaler

HRV = ["rmssd_ms", "sdnn_ms"]
EYE = ["blink_rate_per_minute", "ear_left_mean", "ear_left_std", "ear_right_mean",
       "ear_right_std", "iris_size_estimate_mean", "iris_size_estimate_std", "iris_size_delta_mean"]
CORE = HRV + EYE
ID = ["participant_id", "episode_id"]
SEED = 42


def read_bundle(root, case):
    proxy = case == "v3"
    package = root / ("v3_behavioral_proxy" if proxy else "v4_self_reported_rating")
    target = "cognitive_load_label" if proxy else "difficulty_rating"
    labels = ["Low", "Medium", "High"] if proxy else list(range(1, 11))
    window_dir = Path("window_15s_step_5s")
    hashes = {}

    def read(relative, columns):
        path = package / relative
        hashes[str(Path(relative).as_posix())] = hashlib.sha256(path.read_bytes()).hexdigest()
        return pd.read_csv(path, usecols=columns)

    def attach(left, right, keys, validation):
        merged = left.merge(right, on=keys, how="left", validate=validation, indicator=True)
        if not merged.pop("_merge").eq("both").all():
            raise ValueError(f"CSV pendamping tidak cocok pada {keys}.")
        return merged

    keys = ["window_id", "participant_id", "session_id", target]
    data = read(window_dir / "good_acceptable.csv", keys + CORE)
    assignments = read(window_dir / "window_assignments.csv",
                       keys + ["episode_id"] + (["split"] if proxy else []))
    for table in (data, assignments):
        if table.window_id.duplicated().any() or table[keys].isna().any().any():
            raise ValueError("Kunci window tidak lengkap atau window_id berulang.")
    data = attach(data, assignments, keys, "one_to_one")
    evidence = read("episode_labels.csv" if proxy else "episode_ratings.csv",
                    ID + ["session_id", target])
    if evidence.episode_id.isna().any() or evidence.episode_id.duplicated().any():
        raise ValueError("episode_id pada bukti harus unik dan lengkap.")
    data = attach(data, evidence, ID + ["session_id", target], "many_to_one")
    if set(data[target]) != set(labels) or not np.isfinite(data[CORE].to_numpy(float)).all():
        raise ValueError("Periksa kelas target dan kelengkapan sepuluh fitur core.")
    if proxy:
        splits = read("participant_splits.csv", ["participant_id", "split"])
        if splits.participant_id.duplicated().any() or set(splits.split) != {"Train", "validation", "Test"}:
            raise ValueError("Periksa pembagian peserta v3.")
        data = attach(data, splits, ["participant_id", "split"], "many_to_one")
        weights = read(window_dir / "train_sample_weights.csv", ["window_id", *ID, target, "sample_weight"])
        train = data.split.eq("Train")
        if set(weights.window_id) != set(data.loc[train, "window_id"]) or weights.window_id.duplicated().any():
            raise ValueError("Daftar bobot berbeda dari window Train.")
        checked = attach(data.loc[train], weights, ["window_id", *ID, target], "one_to_one")
        if not np.isfinite(checked.sample_weight).all() or not checked.sample_weight.gt(0).all():
            raise ValueError("Bobot Train harus positif dan finite.")
        data["sample_weight"] = data.window_id.map(checked.set_index("window_id").sample_weight)
    return data.rename(columns={target: "actual"}), labels, hashes

def scores(table, labels):
    actual, predicted = table.actual, table.predicted
    row = dict(n=len(table), participants=table.participant_id.nunique(),
               accuracy=accuracy_score(actual, predicted),
               macro_f1=f1_score(actual, predicted, labels=labels, average="macro", zero_division=0))
    if len(labels) == 10:
        distance = (actual - predicted).abs()
        row.update(mae=distance.mean(), within_one=distance.le(1).mean())
    return row

def save_json(path, value):
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")

## Riwayat sebelum soal dan fitting per peserta

In [ ]:
import itertools
from bisect import bisect_right
from collections import defaultdict
LEGACY_POLICY = 'behavioral_concordance_v1'
POLICY = 'behavioral_concordance_v2'
MIN_REFERENCE = 5

def make_reference(episodes, splits, *, policy=POLICY):
    """One first-exposure observation per person/item; only successful Train cases."""
    seen, items = set(), defaultdict(list)
    for row in sorted(episodes, key=lambda r: (r['start_ms'], r['episode_id'])):
        identity = row['participant_id'], row['item_key']
        if identity in seen:
            continue
        seen.add(identity)
        if (splits[row['participant_id']] == 'Train' and row['valid'] and row['completed']
                and (policy == LEGACY_POLICY or row.get('reference_eligible', True))
                and row['success_submission_type'] == 'manual' and row['active_seconds'] > 0):
            items[row['item_key']].append({'participant_id': row['participant_id'],
                'episode_id': row['episode_id'], 'active_seconds': row['active_seconds']})
    return {'policy': policy, 'minimum_other_people': MIN_REFERENCE, 'quantiles': [1/3,2/3],
            'quantile_method': 'linear', 'source_split': 'Train', 'self_reference': False,
            'items': {key: sorted(values, key=lambda v: v['participant_id']) for key, values in sorted(items.items())}}

def reference_for(reference, episode):
    values = [v for v in reference['items'].get(episode['item_key'], [])
              if v['participant_id'] != episode['participant_id']]
    result = {'participants': [v['participant_id'] for v in values], 'count': len(values),
                  'lower_seconds': None, 'upper_seconds': None, 'reason': ''}
    if len(values) < MIN_REFERENCE:
        result['reason'] = 'insufficient_reference'
    else:
        lower, upper = np.quantile([v['active_seconds'] for v in values], [1/3,2/3], method='linear')
        result.update(lower_seconds=float(lower), upper_seconds=float(upper))
        if not 0 < lower < upper:
            result['reason'] = 'degenerate_reference'
    return result

def label_episode(episode, reference):
    limits = reference_for(reference, episode)
    result = {'cognitive_load_label': '', 'label_reason': '', 'time_band': '', 'performance_band': '',
                  'reference_count': limits['count'], 'reference_participants': limits['participants'],
                  'lower_seconds': limits['lower_seconds'], 'upper_seconds': limits['upper_seconds']}
    if not episode['valid']:
        result['label_reason'] = 'invalid_event_trace'
    elif limits['reason']:
        result['label_reason'] = limits['reason']
    else:
        seconds, errors = episode['active_seconds'], episode['manual_errors']
        time_band = ('Low' if seconds <= limits['lower_seconds'] else
                     'Medium' if seconds <= limits['upper_seconds'] else 'High')
        performance = ('Low' if episode['completed'] and errors == 0 and episode['manual_submissions'] == 1
                       and episode['success_submission_type'] == 'manual' else
                       'Medium' if episode['completed'] and 1 <= errors <= 2 else 'High' if errors >= 3 else '')
        result.update(time_band=time_band, performance_band=performance)
        # An unfinished duration is only a lower bound: never use it for Low/Medium.
        if performance == time_band and (episode['completed'] or time_band == 'High'):
            result.update(cognitive_load_label=time_band, label_reason='concordant_behavioral_evidence')
        else:
            result['label_reason'] = 'conflicting_or_insufficient_behavior'
    return result

class WindowIndex:
    """Match only continuous active time from one episode, without rounding gaps."""

    def __init__(self, episodes):
        self.sessions = defaultdict(list)
        for episode in episodes:
            merged = []
            for start, end in sorted(episode['intervals']):
                if end <= start:
                    raise ValueError('Invalid active interval.')
                if merged and start <= merged[-1][1]:
                    merged[-1][1] = max(end, merged[-1][1])
                else:
                    merged.append([start,end])
            self.sessions[episode['session_id']].extend((start,end,episode) for start,end in merged)
        self.starts = {}
        for session, intervals in self.sessions.items():
            intervals.sort(key=lambda r: r[0])
            if any(b[0] < a[1] for a,b in itertools.pairwise(intervals)):
                raise ValueError('Overlapping active episodes.')
            self.starts[session] = [r[0] for r in intervals]

    def match(self, session_id, start, end):
        if end <= start:
            raise ValueError('Invalid window interval.')
        i = bisect_right(self.starts.get(session_id, []), start)-1
        if i >= 0:
            _, stop, episode = self.sessions[session_id][i]
            if end <= stop:
                return episode, ''
        return None, 'not_single_active_episode'

def epoch_ms(value):
    timestamp = pd.Timestamp(value)
    if pd.isna(timestamp) or timestamp.tzinfo is None:
        raise ValueError(f"Expected a timezone-aware timestamp: {value!r}")
    return timestamp.timestamp() * 1000

HEAD = ["head_pitch_mean", "head_pitch_std", "head_yaw_mean", "head_yaw_std"]
SENSORS = CORE + HEAD



def prior_reference(stream, start):
    # Newest first. Work only with disjoint windows that finished before this task.
    chosen = []
    limit = start
    if stream is not None:
        for index, left, right in stream[["window_start", "window_end"]].itertuples():
            if right <= limit:
                chosen.append(index)
                limit = left
                if len(chosen) == 12:
                    break
    if len(chosen) < 3:
        return pd.Series(np.nan, index=SENSORS), len(chosen), None
    return stream.loc[chosen, SENSORS].median(), len(chosen), stream.at[chosen[0], "window_end"]

def binary(y):
    return y.where(y.eq("High"), "Non-High")

def snapshot_hash(path, expected_lf_hash):
    data = path.read_bytes()
    if hashlib.sha256(data.replace(b"\r\n", b"\n")).hexdigest() != expected_lf_hash:
        raise ValueError(f"Gunakan snapshot v2 15 detik dari repo untuk exp07: {path.name}")
    return hashlib.sha256(data).hexdigest()

def build_pool(bank, all_windows, records, official):
    # Use all quality-eligible sensor windows, including episodes unlabeled by the full-Train reference.
    streams = history_streams(bank)
    frame = pd.read_csv(all_windows, usecols=["window_id", "session_id", "participant_id", "window_start",
                                             "window_end", "window_phase_mix", "quality_tier"] + SENSORS)
    frame = frame[frame.quality_tier.isin(["good", "acceptable"])]
    matcher = WindowIndex(records)
    matches = []
    for row in frame.itertuples():
        episode, _ = matcher.match(row.session_id, epoch_ms(row.window_start), epoch_ms(row.window_end))
        active = all(p["study_phase"] == "active_task" for p in json.loads(row.window_phase_mix)
                     if p["duration_seconds"] > 0)
        if episode and active:
            if episode["participant_id"] != row.participant_id:
                raise ValueError("Pemetaan peserta berbeda.")
            matches.append((row.Index, episode["episode_id"]))
    mapped = frame.loc[[i for i, _ in matches]].copy()
    mapped["episode_id"] = [e for _, e in matches]
    known = pd.DataFrame(records).set_index("episode_id")
    published = mapped[mapped.episode_id.map(known.cognitive_load_label).ne("")]
    cols = ["window_id", "participant_id", "episode_id"]
    pd.testing.assert_frame_equal(published[cols].sort_values(cols).reset_index(drop=True),
                                  official[cols].sort_values(cols).reset_index(drop=True), check_dtype=False)
    # No Test feature vectors enter model development.
    dev = mapped[mapped.episode_id.map(known.split).isin(["Train", "validation"])]
    if not np.isfinite(dev[SENSORS]).all().all():
        raise ValueError("Fitur sensor tidak lengkap.")
    x = dev.groupby("episode_id")[SENSORS].median()
    meta = known.loc[x.index, ["participant_id", "session_id", "start_ms", "split"]]
    refs = []
    for row in meta.itertuples():
        stream = streams.get((row.participant_id, row.session_id))
        start = pd.to_datetime(row.start_ms, unit="ms", utc=True)
        ref, _, _ = prior_reference(stream, start)
        refs.append(ref)
    return x, pd.DataFrame(refs, index=x.index), meta

def fold_labels(records, fit_people):
    roles = {row["participant_id"]: ("Train" if row["participant_id"] in fit_people else "held")
             for row in records}
    reference = make_reference(records, roles)
    return pd.Series({r["episode_id"]: label_episode(r, reference)["cognitive_load_label"]
                      for r in records})

def fold_data(x, refs, meta, records, fitting, held):
    if fitting & held:
        raise ValueError("Peserta fitting dan evaluasi tumpang tindih.")
    labels = fold_labels(records, fitting).reindex(x.index)
    fit_ids = labels.index[labels.ne("") & meta.participant_id.isin(fitting)]
    held_ids = labels.index[labels.ne("") & meta.participant_id.isin(held)]
    y = binary(labels.loc[fit_ids])
    people = meta.participant_id.loc[fit_ids]
    w = 1 / (people.groupby(y).transform("nunique") * y.groupby([y, people]).transform("size"))
    w /= w.mean()
    relative = x - refs.fillna(x.loc[fit_ids].median())
    scaler = StandardScaler().fit(relative.loc[fit_ids], sample_weight=w)
    def scaled(ids):
        return pd.DataFrame(scaler.transform(relative.loc[ids]), index=ids, columns=x.columns)
    if not len(held_ids):
        return None
    results = meta.loc[held_ids, ["participant_id"]].assign(original=labels[held_ids], actual=binary(labels[held_ids]))
    return scaled(fit_ids), y, w, scaled(held_ids), results

def high_score(model, x):
    return model.decision_function(x) * (1 if model.classes_[1] == "High" else -1)

"""Compare three estimators on identical held-out v3 episodes."""

from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

LABELS = ["Non-High", "High"]
MODEL_PARAMS = {
    "svm_rbf": {"C": 1, "kernel": "rbf", "gamma": "scale"},
    "logistic_l2": {"C": 1, "solver": "lbfgs", "max_iter": 2000},
    "extra_trees": {"n_estimators": 300, "max_depth": 3, "min_samples_leaf": 3, "random_state": 42},
}
ESTIMATORS = {"svm_rbf": SVC, "logistic_l2": LogisticRegression, "extra_trees": ExtraTreesClassifier}
THRESHOLDS = {"svm_rbf": 0.0, "logistic_l2": 0.5, "extra_trees": 0.5}


def history_streams(path):
    keys = ["participant_id", "session_id"]
    times = ["window_start", "window_end"]
    bank = pd.read_csv(path, usecols=keys + times + ["quality_tier"] + SENSORS)
    bank = bank[bank.quality_tier.isin(["good", "acceptable"]) & np.isfinite(bank[SENSORS]).all(axis=1)].copy()
    for field in times:
        bank[field] = pd.to_datetime(bank[field], utc=True, format="mixed", errors="raise")
    if bank[keys + times].isna().any().any() or not bank.window_end.gt(bank.window_start).all():
        raise ValueError("Interval sensor historis tidak valid.")
    return dict(tuple(bank.sort_values("window_end", ascending=False).groupby(keys)))


def load_cohort(root):
    official, _, hashes = read_bundle(root, "v3")
    package = root / "v3_behavioral_proxy"
    evidence = pd.read_csv(package / "episode_labels.csv", keep_default_na=False, float_precision="round_trip")
    evidence["intervals"] = evidence.intervals.map(json.loads)
    records = evidence.to_dict("records")
    people = pd.read_csv(package / "participant_splits.csv")
    train = frozenset(people.loc[people.split.eq("Train"), "participant_id"])
    val = frozenset(people.loc[people.split.eq("validation"), "participant_id"])
    if not evidence.episode_id.map(fold_labels(records, train)).eq(evidence.cognitive_load_label).all():
        raise ValueError("Label split resmi berbeda.")
    pool = root / "v2_session_phase/window_15s_step_5s/all_data.csv"
    hashes["v2_all"] = snapshot_hash(pool, "acbef9a60f4114de460ee3c57ca94ff5cae0d48b67229d8b14283d173fef1355")
    # The v2 source is shared by episode mapping and history. History applies its own quality-only filter.
    x, refs, meta = build_pool(pool, pool, records, official)
    return x, refs, meta, records, train, val, hashes


def fit_predict(fit_x, y, weights, held_x, endpoint):
    if set(y) != set(LABELS):
        raise ValueError("Fold fitting harus memuat kedua kelas.")
    rows, coefficients = [], []
    for name, estimator in ESTIMATORS.items():
        model = estimator(**MODEL_PARAMS[name]).fit(fit_x, y, sample_weight=weights)
        features = held_x.columns
        if name == "svm_rbf":
            if model.fit_status_ != 0:
                raise ValueError("SVM belum konvergen.")
            score = high_score(model, held_x)
        else:
            score = model.predict_proba(held_x)[:, list(model.classes_).index("High")]
        if name == "logistic_l2":
            sign = 1 if model.classes_[1] == "High" else -1
            coefficients.append(pd.DataFrame({"feature": features, "coefficient_high": model.coef_[0] * sign}))
        rows.append(endpoint.assign(recipe=name, score=score,
                    predicted=np.where(score > THRESHOLDS[name], "High", "Non-High")).reset_index())
    return pd.concat(rows, ignore_index=True), pd.concat(coefficients, ignore_index=True)


def summarize(pred):
    metrics = []
    for (protocol, recipe), g in pred.groupby(["protocol", "recipe"]):
        tn, fp, fn, tp = confusion_matrix(g.actual, g.predicted, labels=LABELS).ravel()
        if not ((tp + fn) and (tn + fp)):
            raise ValueError(f"Kedua kelas dibutuhkan untuk AUC {protocol}.")
        metrics.append(dict(protocol=protocol, recipe=recipe, participants=g.participant_id.nunique(),
                            episodes=len(g), accuracy=(tp + tn) / len(g),
                            balanced_accuracy=balanced_accuracy_score(g.actual, g.predicted),
                            macro_f1=f1_score(g.actual, g.predicted, labels=LABELS, average="macro", zero_division=0),
                            auc=roc_auc_score(g.actual.eq("High"), g.score), tn=tn, fp=fp, fn=fn, tp=tp))
    stats = pd.DataFrame(metrics)
    ranks = stats[stats.protocol.eq("Train_LOPO")].sort_values(
        ["auc", "balanced_accuracy", "recipe"], ascending=[False, False, True])
    return stats, ranks.reset_index(drop=True)


def run(root, output_root):
    x, refs, meta, records, train, val, hashes = load_cohort(root)
    folds = [("Train_LOPO", participant, train - {participant}, {participant})
             for participant in sorted(train)]
    folds.append(("validation", "validation", train, val))
    predictions, coefficients, audit = [], [], []
    for protocol, fold, fitting, held in folds:
        data = fold_data(x, refs, meta, records, fitting, held)
        if data is None:
            audit.append(dict(protocol=protocol, fold=fold, held_episodes=0))
            continue
        fit_x, y, weights, held_x, endpoint = data
        rows, coefs = fit_predict(fit_x, y, weights, held_x, endpoint)
        predictions.append(rows.assign(protocol=protocol, fold=fold))
        coefficients.append(coefs.assign(protocol=protocol, fold=fold))
        audit.append(dict(protocol=protocol, fold=fold, fit_episodes=len(fit_x), held_episodes=len(held_x),
                          held_with_history=int(refs.loc[held_x.index].notna().all(axis=1).sum()),
                          fit_people=meta.loc[fit_x.index, "participant_id"].nunique()))
    pred = pd.concat(predictions, ignore_index=True)
    endpoint_keys = ["protocol", "fold", "episode_id", "participant_id", "actual", "original"]
    endpoints = pred[pred.recipe.eq("svm_rbf")].set_index(endpoint_keys).sort_index()
    if endpoints.index.has_duplicates or pred.duplicated(endpoint_keys + ["recipe"]).any():
        raise ValueError("Prediksi episode berulang.")
    for name in ESTIMATORS:
        pd.testing.assert_index_equal(pred[pred.recipe.eq(name)].set_index(endpoint_keys).sort_index().index,
                                      endpoints.index)
    metrics, ranking = summarize(pred)
    out = output_root / ("models_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8])
    out.mkdir(parents=True, exist_ok=False)
    for name, frame in [("episode_predictions", pred), ("metrics", metrics), ("lopo_ranking", ranking),
                        ("logistic_coefficients", pd.concat(coefficients)), ("folds", pd.DataFrame(audit))]:
        frame.to_csv(out / (name + ".csv"), index=False)
    save_json(out / "manifest.json", {"inputs": hashes, "models": MODEL_PARAMS, "thresholds": THRESHOLDS,
              "features": SENSORS, "history": "v2 all_data; good/acceptable, finite; prior disjoint windows, min 3 max 12",
              "test_evaluated": False, "candidate_order": ["LOPO AUC", "LOPO balanced accuracy", "recipe"],
              "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__})
    print(metrics.to_string(index=False))
    archive = shutil.make_archive(str(out), "zip", root_dir=out.parent, base_dir=out.name)
    print("ZIP:", archive)
    return Path(archive)


## Evaluasi LOPO dan validasi, lalu ekspor ZIP

In [ ]:
archive = run(DATASET_ROOT, OUTPUT_ROOT)